# Fig 6A — mC vs 3C subtype-clustering agreement (ARI)

For each major type, we independently sub-clustered its cells from **CG methylation** and
from **chromatin contacts**, then measured how much the two label sets agree with the
**adjusted Rand index (ARI)**. Low ARI = the two modalities disagree on subtype identity —
the phenomenon Fig 6 dissects. (Source: `clustering/merged/L2_hiconly/summary.ipynb`.)

## 📥 Required input files

- `{ENTEX_ROOT}/L1color.tsv` · _metadata: major-type color/annotation_
- `{ENTEX_ROOT}/clustering/merged/L2_hiconly/{L1}/mergemcg_rocpr.npy` · _mC subtype labels_
- `{ENTEX_ROOT}/clustering/merged/L2_hiconly/{L1}/mergehic_rocpr.npy` · _3C subtype labels_

In [ ]:
# === Reproduction setup — edit ENTEX_ROOT / REF_ROOT for your machine ===
import os, sys
ENTEX_ROOT = os.environ.get('ENTEX_ROOT', '/large_storage/zhoulab/zhoujt/project/ENTEx')
REF_ROOT   = os.environ.get('REF_ROOT',   '/large_storage/zhoulab/ref')
BOOK_ROOT  = os.environ.get('BOOK_ROOT',  f'{ENTEX_ROOT}/analysis/HumanCellEpigenomeAtlas')
sys.path.insert(0, BOOK_ROOT)
os.chdir(f'{ENTEX_ROOT}/analysis')   # original working directory
import repro_guard                    # no-overwrite guard (default: skip existing)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import adjusted_rand_score as ARI

import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.style.use('default')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Helvetica'

indir = f'{ENTEX_ROOT}/clustering/merged/'
L1_meta = pd.read_csv(f'{ENTEX_ROOT}/L1color.tsv', sep='\t', header=0, index_col=0)

In [ ]:
# ARI between the mC-based and 3C-based subtype labels, per major type.
# (Only the two label vectors are needed; the per-type embedding h5ads are not.)
score = {}
for l1 in L1_meta.index:
    l1_name = L1_meta.loc[l1, 'L1_abbr'].replace(' ', '-').replace('/', '_')
    fmc  = f'{indir}L2_hiconly/{l1_name}/mergemcg_rocpr.npy'
    fhic = f'{indir}L2_hiconly/{l1_name}/mergehic_rocpr.npy'
    if not (os.path.exists(fmc) and os.path.exists(fhic)):
        continue
    labelmc = np.load(fmc,  allow_pickle=True)
    label3c = np.load(fhic, allow_pickle=True)
    score[l1] = ARI(labelmc, label3c)

score = pd.Series(score).sort_values()
print(f'{score.shape[0]} major types; lowest ARI:', score.index[0], round(score.iloc[0], 3))

In [ ]:
nct = score.shape[0]
xticks = np.arange(nct)

fig, ax = plt.subplots(figsize=(6, 2), dpi=300)
ax.spines['bottom'].set_position('zero')
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.bar(x=xticks, height=score, color=L1_meta.loc[score.index, 'color'], width=0.8)

ax.set_xticks(xticks)
ax.set_xticklabels(L1_meta.loc[score.index, 'L1_annot'], fontsize=8, rotation=90)
ax.set_xlim([-0.5, nct - 0.5])
ax.set_yticks([0, 0.2, 0.4])
ax.set_yticklabels([0, 0.2, 0.4], fontsize=8)
ax.set_ylabel('ARI', fontsize=8)
fig.tight_layout()
fig.savefig('clustering/merged/L2_hiconly/mc_3c_L2_ARI.pdf', transparent=True)   # Fig 6A